# Backtest 5 · How much do swap assumptions matter?

The D1 setup chosen on the 8-cross universe, TEST 2019 → latest, under worst-side swap, average swap, and a swap-free account.

In [1]:
from dataclasses import dataclass, replace
from dataclasses import replace
from itertools import product
from pathlib import Path
import argparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
pd.set_option("display.width", 200)
ROOT = Path.cwd().parent   # repository root (run the notebook from its own folder); data lives in ROOT / "data"

### Forex PCA residual model

In [2]:
CCYS = ["EUR", "GBP", "AUD", "NZD", "CAD", "CHF", "JPY", "NOK", "SEK", "XAU", "XAG"]

CROSSES_8 = {  # symbol: (base, quote)
    "AUDNZD": ("AUD", "NZD"), "EURGBP": ("EUR", "GBP"), "EURCHF": ("EUR", "CHF"),
    "AUDCAD": ("AUD", "CAD"), "NZDCAD": ("NZD", "CAD"), "CADCHF": ("CAD", "CHF"),
    "EURNOK": ("EUR", "NOK"), "EURSEK": ("EUR", "SEK"),
}

# all 23 crosses downloaded (data_collection/01): the 8 above + the rest of the 28 pairs without USD
CROSSES_ALL = {**CROSSES_8, **{s: (s[:3], s[3:]) for s in [
    "EURJPY", "GBPJPY", "AUDJPY", "NZDJPY", "CADJPY", "CHFJPY", "EURAUD", "EURNZD",
    "EURCAD", "GBPAUD", "GBPNZD", "GBPCAD", "GBPCHF", "AUDCHF", "NZDCHF"]}}

# HistData has no spreads: assumed typical prop-firm raw spreads, as % of price (conservative)
SPREAD_PCT = {"AUDNZD": 0.018, "EURGBP": 0.012, "EURCHF": 0.016, "AUDCAD": 0.016,
              "NZDCAD": 0.024, "CADCHF": 0.025, "EURNOK": 0.035, "EURSEK": 0.036}

COMMISSION_PCT = 0.003  # per side, ~ $6 per 100k round turn
BARS_PER_YEAR = 5900  # hourly bars actually present per year in the data
CROSSES = CROSSES_8   # active universe; later cells may switch it to CROSSES_ALL
SWAP = "worst"        # daily / 4h tests: "worst" = charge the more expensive swap side, "none" = swap-free

In [3]:
@dataclass(frozen=True)
class Params:
    window: int = 480
    step: int = 24
    k: int = 3                 # PCA factors removed
    entry: float = 2.0
    exit: float = 0.5
    stop: float = 4.0
    max_hl: float = 48.0
    max_hold: int = 96
    target_vol: float = 0.05   # annual vol per open position
    max_lev: float = 30.0
    spread_mult: float = 1.0
    real_spreads: bool = False  # True: per-bar spreads recorded by Vantage MT5 (data/mt5_fx)
    bars_per_year: float = BARS_PER_YEAR  # 5900 for H1; set for other timeframes
    vol_lookback: int = 480     # bars used to size positions

In [4]:
def model_signals(r, pairs, prm):
    """Out-of-sample residuals and s-scores for each cross. r: (T, N) log returns of CCYS."""
    T = len(r)
    P = len(pairs)
    s = np.full((T, P), np.nan)
    hl = np.full((T, P), np.nan)
    resid = np.full((T, P), np.nan)
    explained = np.full(T, np.nan)
    for t0 in range(prm.window, T, prm.step):
        w = r[t0 - prm.window:t0]
        mu, sd = w.mean(0), w.std(0, ddof=1)
        z = (w - mu) / sd
        vals, vecs = np.linalg.eigh(np.corrcoef(z.T))
        o = vals.argsort()[::-1]
        V = vecs[:, o[:prm.k]]
        t1 = min(t0 + prm.step, T)
        explained[t0:t1] = vals[o[0]] / vals.sum()
        e_win = (z - z @ V @ V.T) * sd                         # residual returns, window
        zn = (r[t0:t1] - mu) / sd
        e_new = (zn - zn @ V @ V.T) * sd                       # residual returns, next bars (frozen model)
        for j, (a, b) in enumerate(pairs):
            p_win = e_win[:, a] - e_win[:, b]
            p_new = e_new[:, a] - e_new[:, b]
            resid[t0:t1, j] = p_new
            x = np.cumsum(p_win)
            bb, aa = np.polyfit(x[:-1], x[1:], 1)
            if not 0 < bb < 1:
                continue
            zeta = x[1:] - (aa + bb * x[:-1])
            m = aa / (1 - bb)
            sig = np.sqrt(zeta.var() / (1 - bb * bb))
            s[t0:t1, j] = (x[-1] + np.cumsum(p_new) - m) / sig
            hl[t0:t1, j] = np.log(2) / -np.log(bb)
    return s, hl, resid, explained

In [5]:
def backtest_one(px, s, hl, idx, cost_side, prm, carry=0.0):
    """One cross. px: closes, s/hl: signals at each bar close, cost_side: cost per side (scalar or per bar),
    carry: holding cost per bar while a position is open (swap), as a fraction of notional."""
    cost_side = np.broadcast_to(np.asarray(cost_side, dtype=float), (len(px),))
    carry = np.broadcast_to(np.asarray(carry, dtype=float), (len(px),))
    simple = np.r_[0.0, px[1:] / px[:-1] - 1]
    lr = np.r_[0.0, np.diff(np.log(px))]
    bar_target = prm.target_vol / np.sqrt(prm.bars_per_year)
    n = len(px)
    ret, gross = np.zeros(n), np.zeros(n)
    pos, lev, held, blocked, cur, trades = 0, 0.0, 0, 0, None, []
    for t in range(n):
        if pos != 0:
            g = pos * lev * simple[t]
            ret[t] += g - lev * carry[t]
            gross[t] += g
            cur["gross"] += g
            cur["cost"] += lev * carry[t]
            held += 1
        st = s[t]
        want, reason = pos, None
        if pos != 0:
            if np.isnan(st):
                want, reason = 0, "model invalid"
            elif (pos > 0 and st >= -prm.exit) or (pos < 0 and st <= prm.exit):
                want, reason = 0, "target"
            elif abs(st) >= prm.stop:
                want, reason = 0, "stop"
            elif held >= prm.max_hold:
                want, reason = 0, "time"
        if blocked and (np.isnan(st) or abs(st) < prm.entry):
            blocked = 0
        if want == 0 and pos == 0 and not blocked and not np.isnan(st) \
                and hl[t] <= prm.max_hl and prm.entry <= abs(st) < prm.stop:
            want = -1 if st > 0 else 1
        if want != pos:
            if pos != 0:
                ret[t] -= lev * cost_side[t]
                cur.update(cost=cur["cost"] + lev * cost_side[t], exit=idx[t], bars=held, reason=reason)
                trades.append(cur)
                blocked = 1 if reason in ("stop", "time") else 0
            if want != 0:
                vol = lr[max(0, t - prm.vol_lookback):t + 1].std()
                lev = min(bar_target / vol, prm.max_lev) if vol > 0 else 0.0
                ret[t] -= lev * cost_side[t]
                cur = {"entry": idx[t], "dir": want, "s_entry": st, "half_life": hl[t], "lev": lev,
                       "gross": 0.0, "cost": lev * cost_side[t]}
                held = 0
            pos = want
    tr = pd.DataFrame(trades)
    if len(tr):
        tr["net"] = tr["gross"] - tr["cost"]
    return ret, gross, tr

In [6]:
def variance_ratio(x, k):
    x = pd.Series(x).dropna()
    agg = x.rolling(k).sum().iloc[k - 1::k]
    return agg.var() / (k * x.var())

### Daily / 4-hour candles, costs and walk-forward engine

In [7]:
SPLIT = pd.Timestamp("2019-01-01", tz="UTC")

TF = {
    # bars per trading day, bars per year, model re-fit step, grid
    "D1": {"per_day": 1, "per_year": 260, "step": 5,
           "grid": {"window": [60, 120, 250], "entry": [1.5, 2.0, 2.5], "exit": [0.0, 0.5], "max_hl": [5, 10, 20]}},
    "H4": {"per_day": 6, "per_year": 1560, "step": 6,
           "grid": {"window": [180, 360, 720], "entry": [1.5, 2.0, 2.5], "exit": [0.0, 0.5], "max_hl": [12, 30, 60]}},
}

In [8]:
def candle_key(idx, tf):
    """Group hourly UTC timestamps into candles that start at 17:00 New York (MT5 broker convention)."""
    shifted = idx.tz_convert("America/New_York").tz_localize(None) + pd.Timedelta(hours=7)
    return shifted.floor("D" if tf == "D1" else "4h")

In [9]:
def to_candles(hourly, tf, how="last"):
    key = candle_key(hourly.index, tf)
    g = hourly.groupby(key)
    out = g.last() if how == "last" else g.median()
    stamps = hourly.index.to_series().groupby(key).last()  # stamp = last hour in the candle (UTC)
    out.index = pd.DatetimeIndex(stamps.reindex(out.index).to_list(), name="time_utc")
    return out

In [10]:
def load(tf):
    d = ROOT / "data/histdata_fx"
    cur_h = pd.read_csv(d / "currencies_1h.csv", index_col=0, parse_dates=True)[CCYS]
    crs_h = pd.read_csv(d / "crosses_1h.csv", index_col=0, parse_dates=True)[list(CROSSES)]
    for sym, (a, b) in CROSSES.items():  # gaps in a cross's own file -> rebuild from the majors
        crs_h[sym] = crs_h[sym].fillna(cur_h[a] / cur_h[b])
    cur = to_candles(cur_h.ffill(limit=3), tf).dropna()
    crs = to_candles(crs_h.ffill(limit=3), tf).reindex(cur.index).ffill()

    # spreads: Vantage hourly spreads -> median per candle; earlier years -> each cross's overall median
    sp_h = pd.read_csv(ROOT / "data/mt5_fx/crosses_1h.csv", index_col=0, parse_dates=True)
    sp_h = sp_h[[f"{s}_spread" for s in CROSSES]]
    sp_h.columns = list(CROSSES)
    sp = to_candles(sp_h, tf, how="median").reindex(cur.index)
    sp = sp.fillna(sp_h.median())

    swaps = pd.read_csv(ROOT / "data/mt5_fx/swaps.csv", index_col=0)
    worst = (-swaps[["long %/night", "short %/night"]].astype(float)).max(axis=1).clip(lower=0) / 100
    per_bar_swap = worst.reindex(list(CROSSES)) * 7 / 5 / TF[tf]["per_day"] * (SWAP == "worst")  # 7 nights paid over 5 trading days
    return cur, crs, sp, per_bar_swap

In [11]:
def run_tf(cur, crs, sp, swap, prm, sig=None):
    r = np.log(cur[CCYS]).diff().to_numpy()[1:]
    idx = cur.index[1:]
    pairs = [(CCYS.index(a), CCYS.index(b)) for a, b in CROSSES.values()]
    if sig is None:
        sig = model_signals(r, pairs, prm)
    s, hl, *_ = sig
    per = {}
    for j, sym in enumerate(CROSSES):
        cost = sp[sym].reindex(idx).to_numpy() / 2 * prm.spread_mult + COMMISSION_PCT / 100
        ret, gross, tr = backtest_one(crs[sym].reindex(idx).to_numpy(), s[:, j], hl[:, j], idx, cost, prm,
                                      carry=swap[sym])
        per[sym] = (ret, gross, tr)
    return per, sig, idx

In [12]:
def perf(ret, per_year):
    ret = pd.Series(ret)
    if len(ret) == 0 or ret.std() == 0:
        return {"Sharpe": np.nan, "%/yr": 0.0, "maxDD %": 0.0}
    eq = (1 + ret).cumprod()
    return {"Sharpe": ret.mean() / ret.std() * np.sqrt(per_year), "%/yr": ret.mean() * per_year * 100,
            "maxDD %": (eq / eq.cummax() - 1).min() * 100}

In [13]:
def combine(per, syms, idx, mask, k=0):
    arr = sum(per[s][k] for s in syms) if syms else np.zeros(len(idx))
    return pd.Series(arr[mask], index=idx[mask])

In [14]:
def trades(per, syms, lo, hi):
    t = [per[s][2].assign(symbol=s) for s in syms if len(per[s][2])]
    t = pd.concat(t, ignore_index=True) if t else pd.DataFrame(columns=["entry", "net", "gross", "cost", "bars"])
    return t[(t["entry"] >= lo) & (t["entry"] < hi)]

## Swap scenarios

In [15]:
CHOSEN = dict(window=120, entry=2.5, exit=0.5, max_hl=10)
CROSSES_KEPT = ["AUDNZD", "EURGBP", "AUDCAD", "NZDCAD", "EURNOK", "EURSEK"]

cur, crs, sp, worst = load("D1")
swaps = pd.read_csv(ROOT / "data/mt5_fx/swaps.csv", index_col=0)
avg = (-(swaps["long %/night"].astype(float) + swaps["short %/night"].astype(float)) / 2).clip(lower=0) / 100 * 7 / 5
cfg = TF["D1"]
prm = replace(Params(), step=cfg["step"], max_hold=30, bars_per_year=cfg["per_year"],
              vol_lookback=CHOSEN["window"], **CHOSEN)

sig = None
print("D1 chosen setup, TEST 2019-latest:")
for label, sw in [("worst-side swap (used in walk-forward)", worst), ("average of long/short swap", avg),
                  ("swap-free account", worst * 0)]:
    per, sig, idx = run_tf(cur, crs, sp, sw, prm, sig)
    test = idx >= SPLIT
    p = perf(combine(per, CROSSES_KEPT, idx, test), cfg["per_year"])
    t = trades(per, CROSSES_KEPT, SPLIT, idx[-1] + pd.Timedelta(days=1))
    print(f"  {label:40s} Sharpe {p['Sharpe']:5.2f}  {p['%/yr']:5.2f}%/yr  maxDD {p['maxDD %']:6.2f}%  "
          f"avg cost/trade {t['cost'].mean() * 100:.3f}%  (avg gross {t['gross'].mean() * 100:.3f}%)")

D1 chosen setup, TEST 2019-latest:


  worst-side swap (used in walk-forward)   Sharpe  0.15   0.72%/yr  maxDD -12.19%  avg cost/trade 0.115%  (avg gross 0.158%)


  average of long/short swap               Sharpe  0.30   1.45%/yr  maxDD -10.56%  avg cost/trade 0.060%  (avg gross 0.158%)


  swap-free account                        Sharpe  0.41   1.97%/yr  maxDD  -9.42%  avg cost/trade 0.020%  (avg gross 0.158%)


**Result:** the daily trades earn about 0.15 – 0.35% each and are held ≈ 11 days, so swap is the deciding cost: the strategy needs a swap-free or low-swap account.